# Exercise 05c — OD generation, interpretation, and scenarios

This exercise treats OD generation as a spatial generalisation problem. Complete origin zones are held out, so the model must generate destination flows for places whose historical OD vector was not used for fitting.

The data preparation and origin-based split are provided. Tune features and models with validation origins, compare all frozen families once on test origins, then use maps and scenario analysis to interpret the result.

## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `05 Transportation Applications I`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "05 Transportation Applications I"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn


## Provided data preparation

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
np.random.seed(42)
od = pd.read_csv("nyc_static_od.csv.gz")
zones = pd.read_csv("nyc_zone_features.csv")
geo = json.load(open("nyc_taxi_zones.geojson"))
train = od.split.eq("train")
valid = od.split.eq("validation")
test = od.split.eq("test")
y = od.share.fillna(0).to_numpy()

gravity = ["distance_km", "origin_population", "destination_population", "origin_employed_residents", "destination_employed_residents"]
urban = gravity + [column for column in od if any(column.endswith(suffix) for suffix in ["commercial_landuse", "industrial_landuse", "residential_landuse", "retail_landuse", "main_road", "other_road", "residential_road"])]
full = urban + [column for column in od if column.endswith("_facilities")]
feature_sets = {"gravity variables": gravity, "+ urban form": urban, "+ POIs": full}

X_train = od.loc[train, full].to_numpy()
y_train = y[train]
X_valid = od.loc[valid, full].to_numpy()
y_valid = y[valid]
X_test = od.loc[test, full].to_numpy()
y_test = y[test]
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)
print("Each row is one origin–destination candidate; y is destination share.")

def normalise_scores(frame, scores):
    result = frame[["origin_id", "destination_id", "origin_Zone", "destination_Zone", "origin_outflow", "flow"]].copy()
    result["score"] = np.clip(scores, 0, None)
    total = result.groupby("origin_id").score.transform("sum")
    count = result.groupby("origin_id").score.transform("size")
    result["pred_share"] = np.where(total > 0, result.score / total, 1 / count)
    result["pred_flow"] = result.pred_share * result.origin_outflow
    return result

def cpc(frame):
    return 2 * np.minimum(frame.flow, frame.pred_flow).sum() / np.maximum((frame.flow + frame.pred_flow).sum(), 1)

def evaluate(mask, model, columns):
    generated = normalise_scores(od.loc[mask], model.predict(od.loc[mask, columns]))
    return generated, {
        "MAE": mean_absolute_error(generated.flow, generated.pred_flow),
        "CPC": cpc(generated),
    }

def polygons(feature):
    geometry = feature["geometry"]
    return geometry["coordinates"] if geometry["type"] == "MultiPolygon" else [geometry["coordinates"]]

def zone_map(values, title, ax=None, cmap="YlGnBu"):
    values = pd.Series(values)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    norm = plt.Normalize(values.min(), values.max() if values.max() > values.min() else values.min() + 1)
    for feature in geo["features"]:
        zone_id = int(feature["properties"]["LocationID"])
        value = values.get(zone_id, np.nan)
        colour = "lightgrey" if pd.isna(value) else plt.colormaps[cmap](norm(value))
        for polygon in polygons(feature):
            ax.fill(*np.asarray(polygon[0]).T, facecolor=colour, edgecolor="white", linewidth=.45)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=.7)
    return ax

## Question 1 — Select geographic feature groups

Using a Ridge model with `alpha=1`, compare the three nested feature sets on validation origins. Select the set with the highest validation CPC and explain what CPC measures that row-wise share MAE does not.

In [ ]:
# TODO: fit one Ridge model per feature set and evaluate validation CPC.
# Store selected_features_name and selected_features.

## Question 2 — Tune Random Forest and ANN

Using the selected features, tune Random Forest depth in `[4, 8, None]` and ANN width in `[32, 64]`. For the ANN, scale destination shares to percentage points during fitting as in the tutorial. Select one configuration per family using validation CPC.

In [ ]:
# TODO: tune forest depth and ANN width using validation CPC.
# Store one selected key per family in best_keys.

## Question 3 — Compare all frozen families on unseen origins

Tune Ridge `alpha` in `[0.1, 1, 10]` on validation CPC, then refit the selected Ridge, forest, and ANN on train + validation origins. Evaluate all three once on test origins using flow MAE and CPC.

In [ ]:
# TODO: tune Ridge alpha on validation CPC.
# TODO: refit selected Ridge, forest, and ANN on train + validation and evaluate test once.

## Question 4 — Explain one held-out origin and inspect feature importance

For the test-best model, select the held-out origin with the largest observed outflow. Plot observed and generated destination flows on two maps and tabulate its ten largest absolute errors. For the forest, draw the 15 largest feature importances and distinguish predictive association from causal effect.

In [ ]:
# TODO: map observed and predicted destinations for one held-out origin.
# TODO: tabulate its largest errors and draw forest feature importance.

## Question 5 — Extension: a planning scenario

Choose one destination zone and increase its destination employment by 20% in a copy of the test feature table. Use the frozen best model to regenerate flows, then compare the selected destination's predicted inflow before and after the change. Explain why this is a model-based scenario response rather than a causal forecast.

In [ ]:
# TODO: modify destination employment in a copy of the test rows.
# TODO: regenerate origin-normalised flows and compare destination inflow before/after.